# PA005 - High Value Customer Identification (Insiders)

Next steps: Create/update function to apply all data processing steps and
- Clustering the customers through the RFM Matrix
- Clustering the customers using ML Algorithm
    
    CREATE A NEW NOTEBOOK AND SEE IF THE CLUSTER IMPROVES, BECAUSE I GOT ONE WITH 31 CUSTOMERS THAT HAVE SOLD 2MM.

# 0 - IMPORTS

In [1]:
# Data Maniputalion and Data Analysis
import re
import warnings

import pandas                as pd
import numpy                 as np
import plotly.express        as px
import plotly.graph_objects  as go
import matplotlib.cm         as cm
import matplotlib            as mpl

from plotly.offline          import iplot
from matplotlib              import pyplot          as plt


# Loading Images and display settings
from IPython.display         import Image, display
from IPython.core.display    import HTML

warnings.filterwarnings( 'ignore' )

## 0.1 - Helper Functions

In [7]:
def drop_and_rename_duplicate_columns(df):
    """
    Remove colunas duplicadas resultantes de um merge e renomeia as colunas para eliminar o sufixo '_x'.
    Remove linhas referentes a invoices canceladas ou devolvidas: cujo valor para a coluna `invoice_status_y` for igual a 'True'.
    
    Parâmetros:
    - df (pd.DataFrame): DataFrame resultante de um merge com possíveis colunas duplicadas.

    Retorna:
    - pd.DataFrame: DataFrame com colunas duplicadas removidas e renomeadas.
    """

    # Verifica se a coluna 'invoice_status_y' existe e filtra onde o valor é True
    if 'invoice_cancelled_y' in df.columns:
        df = df[~df['invoice_cancelled_y'].fillna(False)]
    
    # Identifica colunas com sufixos '_x' e '_y' após o merge
    duplicate_columns = [col for col in df.columns if col.endswith('_x') or col.endswith('_y')]
    
    # Cria um mapeamento para manter apenas uma ocorrência e renomear as colunas
    cols_to_keep = {}
    for col in duplicate_columns:
        base_name = col[:-2]  # Remove o sufixo '_x' ou '_y'
        if base_name not in cols_to_keep:
            # Guarda a coluna com sufixo '_x' para manter e renomear
            cols_to_keep[base_name] = col

    # Define as colunas para remoção, mantendo apenas uma de cada par duplicado
    cols_to_drop = set(duplicate_columns) - set(cols_to_keep.values())
    df = df.drop(columns=cols_to_drop)

    # Renomeia as colunas restantes, removendo o sufixo '_x'
    df = df.rename(columns={old_name: base_name for base_name, old_name in cols_to_keep.items()})

    # Filtra as colunas, removendo invoice_cancelled
    cols = ['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date','unit_price', 'customer_id', 'country']

    return df[cols]

def data_preparation(df):

    # Rename Columns:
    cols_new = ['invoice_no','stock_code','description','quantity','invoice_date','unit_price','customer_id','country']
    df.columns = cols_new

    # Removing registers with NaN values
    df= df.loc[~df['customer_id'].isna(),:]

    # Changing data types
    df[ 'invoice_date'] = pd.to_datetime( df['invoice_date'], format='%d-%b-%y') # changing the data on the column invoice data to match the correct data type
    
    df['customer_id'] = df['customer_id'].astype( int ) # changing the data on the column customer id data to match the correct data type
    
    df['country'] = df['country'].astype( str ) # Changing data type from object to string

    # Applying filters to remove noise from the data

    #### REMOVING REGISTERS WHERE THE PURCHASE HAVE BEEN RETURNED OR CANCELLED ####
    # Classifying each invoice as cancelled (True) or not (False) 
    df['invoice_cancelled'] = df['invoice_no'].str.startswith("C") & (df['quantity']<0)
    
    # Separting two datasets:
    df_returns = df[df['invoice_cancelled']] # invoices cancelled
    df_purchases = df[~df['invoice_cancelled']] # invoices not cancelled
    
    # Mergin the above two data sets
    merged_df = df_purchases.merge(df_returns, on=['stock_code','unit_price','customer_id'], how='left')
    
    # Applying function to clean the new data set by removing columns etc.
    df = drop_and_rename_duplicate_columns(merged_df)
    
    # Removing the register from customer 15098 which has an extreme value of gross_revenue
    df = df.drop(df.index[146375]).reset_index(drop=True)
    
    # --- NUMERICAL ATTRIBUTES ---
    
    # Filtering products where price is equal to 0
    df = df.loc[df['unit_price'] > 0, :]
    
    # --- CATEGORICAL ATTRIBUTES ---
    
    # Filtering stock_codes that does not reffers to items
    df = df[~df['stock_code'].isin( ['POST','C2','DOT','PADS','BANK CHARGES'] )]
    
    # Description
    df = df.drop( columns='description', axis=1)


    ##### FEATURE ENGINEERING #####
    
    # data reference
    df_ref = df.drop( ['invoice_no','stock_code','quantity','invoice_date','unit_price','country'],axis=1).drop_duplicates(ignore_index=True)
    
    # Calculus of the monetary value sold
    df.loc[:,'gross_revenue'] = df.loc[:,'quantity'] * df.loc[:,'unit_price']
    df_sold = df.loc[:, ['customer_id', 'gross_revenue']].groupby('customer_id').sum().reset_index()

    df_ref = pd.merge( df_ref, df_sold, on='customer_id', how='left')
    
    # Recency
    df_recency = df.loc[:, ['customer_id', 'invoice_date']].groupby('customer_id').max().reset_index()
    df_recency['recency_days'] = (df['invoice_date'].max() - df_recency['invoice_date']).dt.days
    df_recency = df_recency[['customer_id','recency_days']].copy()
    df_ref = pd.merge( df_ref, df_recency, on='customer_id', how='left')

    # Calculate the quantity of invoices issues for an unique customer
    df_freq = (df.loc[:, ['customer_id', 'invoice_no']].drop_duplicates()
                                                    .groupby('customer_id')
                                                    .count()
                                                    .reset_index()
                                                    .rename( columns={'invoice_no':'qty_invoices'}) )
                
    df_ref = pd.merge(df_ref, df_freq, on='customer_id', how='left')

    # Number of products
    df_freqp = (df.loc[:,['customer_id', 'quantity']].groupby('customer_id')
                                                       .sum().reset_index()
                                                       .rename( columns={'quantity':'qty_prod_purchased'}) )

    df_ref = pd.merge(df_ref, df_freqp, on='customer_id', how='left')
   
    # Calculate the range of products per customer
    
    df_prod = ( df.loc[:,['customer_id', 'stock_code']].groupby('customer_id')
                                                    .count()
                                                    .reset_index()
                                                    .rename( columns={'stock_code':'range_of_products'}) )
            
    df_ref = pd.merge(df_ref, df_prod, on='customer_id', how='left')

    # Calculate the average spenses per customer
    df_avg_ticket = ( df.loc[:, ['customer_id','gross_revenue']].groupby('customer_id')
                                                                 .mean()
                                                                 .reset_index()
                                                                 .rename( columns={'gross_revenue':'avg_ticket'}) )

    df_ref = pd.merge( df_ref, df_avg_ticket, on='customer_id', how='left')
    
    # Frequency
    df_max = df[['customer_id','invoice_date']].drop_duplicates().groupby('customer_id').max().reset_index() # finding the last date of purchase per customer
    df_min = df[['customer_id','invoice_date']].drop_duplicates().groupby('customer_id').min().reset_index() # finding the last date of purchase per customer

    df_aux = ( df[['customer_id','invoice_no','invoice_date']].drop_duplicates()
                                                               .groupby('customer_id')
                                                               .agg( max_ =('invoice_date', 'max'),
                                                                     min_ =('invoice_date', 'min'),
                                                                     days_=('invoice_date', lambda x:( (x.max() - x.min() ).days)+1),
                                                                     buy_ =('invoice_no', 'count'))).reset_index()

    # Frequency
    df_aux['frequency'] = df_aux[['buy_', 'days_']].apply( lambda x: x['buy_'] / x['days_'] if x['days_'] != 0 else 0, axis=1 )

    # Merge
    df_ref = pd.merge(df_ref, df_aux[['customer_id','frequency']], on='customer_id', how='left')
    
    # Quantity of products cancelled
    df_returns = df_returns[['customer_id','quantity']].groupby('customer_id').sum().reset_index().rename( columns={'quantity':'qty_returns'})
    df_returns['qty_returns'] = df_returns['qty_returns'] * -1

    df_ref = pd.merge( df_ref, df_returns, on='customer_id', how='left')
    df_ref.loc[df_ref['qty_returns'].isna(), 'qty_returns'] = 0

    # Calculate the average qty of products purchased
    df_ref['avg_qty_products_purchased'] = df_ref['qty_prod_purchased'] / df_ref['qty_invoices']

    # Retrieving the week day for the specific date
    df['week_day']= df['invoice_date'].dt.dayofweek

    # Creating the dataframe with day_week per invoice_no
    aux_02 = df[['invoice_no','customer_id','week_day']].drop_duplicates(ignore_index=True)

    # Calculus of week day most frequent
    aux_03 = aux_02[['customer_id', 'week_day']].groupby('customer_id').apply(lambda x: x.mode().iloc[0]).reset_index(drop=True)

    # Adding the new feature into the data set
    df_ref = pd.merge( df_ref, aux_03, on='customer_id', how='left')

    # Retrieving the week day for the specific date
    df['month']= df['invoice_date'].dt.month

    # Creating the dataframe with day_week per invoice_no
    aux_312 = df[['invoice_no','customer_id','month']].drop_duplicates(ignore_index=True)

    # Calculus of week day most frequent
    aux_3123 = aux_312[['customer_id', 'month']].groupby('customer_id').apply(lambda x: x.mode().iloc[0]).reset_index(drop=True)

    # Adding the new feature into the data set
    df_ref = pd.merge( df_ref, aux_3123, on='customer_id', how='left')

    return(df_ref)

In [8]:
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


# 1 - DATA MANIPULATION

## 1.1 - Loading Data

In [4]:
df = pd.read_csv("../data/raw/Ecommerce.csv")

# drop extra column
df = df.drop( columns=['Unnamed: 8'], axis=1)

## 1.2 - Data Treating

In [ ]:
df1 = data_preparation(df)